# 12. Preprocessing and leakage-safe pipelines

**Level:** Intermediate

**Prerequisites:** 11: splits and training boundaries.

[Course index](00_START_HERE.ipynb) | [Previous](11_Datasets_Supervision_and_Data_Splits.ipynb) | [Next](13_Linear_Regression.ipynb)

**Topics:** Imputation, Scaling, One-hot encoding, ColumnTransformer, Pipeline, Data leakage.

**Matching script:** `examples\12_Preprocessing_and_Leakage_Safe_Pipelines.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A preprocessing pipeline is part of the model, not an optional cleaning step performed on all
data before splitting. Median imputation, scaling statistics, learned vocabularies, and feature
selection can leak information when fitted on validation/test data.

fit learns transform state from training examples; transform applies it to new examples.
A Pipeline keeps preprocessing and estimation together during training and cross-validation.
ColumnTransformer applies different transformations to numeric and categorical columns.
One-hot encoding creates indicator columns without inventing a numeric order between sites.

Unknown categories at inference need an explicit policy. handle_unknown='ignore' yields an
all-zero indicator for unseen categories; that is a design choice, not proof the prediction
remains trustworthy. Detect domain shift and missing inputs separately. Preserve the training
feature order and units at serving time.

## Mathematics or data contract

Standardization uses z=(x-training_mean)/training_std, not statistics recomputed from the test batch.

## Important limitation

The toy training set is for pipeline mechanics; its training accuracy is not a generalization estimate.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Mixed numeric and categorical preprocessing

**Purpose:** Fit transform state only through the training pipeline.

### Read the code in this order
1. Select numeric and site columns by name.
2. Impute missing latency and standardize it.
3. Encode site values and fit logistic regression.
4. Predict an unseen site and verify finite probabilities.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict_proba(...)` returns estimated class probabilities in the estimator's class order.
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
new-site probabilities: [[0.334, 0.666]]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.DataFrame({"latency": [1., 2., np.nan, 8., 9., 10.],
                      "site": ["a", "a", "b", "b", "a", "b"]})
y = [0, 0, 0, 1, 1, 1]
numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
preprocessing = ColumnTransformer([
    ("numeric", numeric, ["latency"]),
    ("site", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["site"]),
])
model = make_pipeline(preprocessing, LogisticRegression())
model.fit(train, y)
new = pd.DataFrame({"latency": [9.], "site": ["new-site"]})
probabilities = model.predict_proba(new)
assert probabilities.shape == (1, 2) and np.isfinite(probabilities).all()
assert np.isclose(probabilities.sum(), 1)
print("new-site probabilities:", np.round(probabilities, 3).tolist())

new-site probabilities: [[0.334, 0.666]]


### Interpret example 1

The result should demonstrate this purpose: **Fit transform state only through the training pipeline.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add a train/test split and compare a leakage-safe pipeline with an intentionally leaked scaler.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Imputation` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Standardization uses z=(x-training_mean)/training_std, not statistics recomputed from the test batch.
- Name the boundary of the demonstration: The toy training set is for pipeline mechanics; its training accuracy is not a generalization estimate.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.